<a href="https://colab.research.google.com/github/wizrads/AVATAR/blob/main/Lab8_WinstonLutz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MP-566 Lab 8 — Winston-Lutz Test

Use [**pylinac**](https://pylinac.readthedocs.io/en/latest/winston_lutz.html) to check whether the TrueBeam's **mechanical** and **radiation** isocenters coincide within TG-142 / TG-198 tolerance.

**What you'll produce**
- Table of per-image 2D shifts (`Δu`, `Δv`, total).
- 3D shift vector from the radiation isocenter to the BB.
- Several pylinac-generated figures (per-image overlays, axis-grouped views, 3D isocenter visualization, summary plot).

Cells labeled **`TODO`** are for you to fill in. Each has a hint.

## 1. Install pylinac

In [ ]:
!pip install --quiet pylinac pandas pydicom
import warnings
warnings.filterwarnings("ignore")

## 2. Upload the DICOMs

Run the cell below and use the file picker to select all **14 `.dcm` files** (one per Winston-Lutz field in Table 1 of the handout). They will be copied into a `Lab_Data_Final/` folder next to the notebook.

> Tip: in the Colab upload dialog you can Ctrl/Cmd-click or Shift-click to select all 14 files at once.

In [ ]:
import os, glob, shutil

DATA_DIR = "Lab_Data_Final"
os.makedirs(DATA_DIR, exist_ok=True)

# If the folder doesn't already have the 14 DICOMs, prompt an upload (Colab only)
if len(glob.glob(os.path.join(DATA_DIR, "*.dcm"))) < 14:
    try:
        from google.colab import files
        print("Please select all 14 .dcm files in the picker ...")
        uploaded = files.upload()
        for fname in uploaded:
            shutil.move(fname, os.path.join(DATA_DIR, fname))
    except ImportError:
        raise FileNotFoundError(
            f"Not on Colab. Place the 14 .dcm files inside '{DATA_DIR}/' next to this notebook."
        )

# Sanity check
n = len(glob.glob(os.path.join(DATA_DIR, "*.dcm")))
print(f"Found {n} DICOM files in '{DATA_DIR}/'")
assert n == 14, f"Expected 14 WL images, got {n}."

## 3. Meet your DICOM files

**DICOM** (Digital Imaging and Communications in Medicine) is the standard file format in radiology and radiation oncology. Every DICOM file contains two things:

1. A **pixel array** (the actual image/radiation data).
2. A **header** of tagged metadata: patient info, machine parameters, beam geometry, acquisition time, etc.

Each tag is identified by a `(group, element)` number and a human-readable keyword (e.g., `(0018, 1110)` = `"RTImageSID"`). Python's [`pydicom`](https://pydicom.github.io/) library lets you read both in one line.

For the Winston-Lutz analysis, the tags that matter are:

| DICOM keyword | What it is | Why we care |
|---|---|---|
| `GantryAngle` | Gantry rotation (°) | Which direction the beam came from |
| `BeamLimitingDeviceAngle` | Collimator angle (°) | Record only |
| `PatientSupportAngle` | Couch rotation (°) | Couch-kick fields (13 & 14) |
| `ImagePlanePixelSpacing` | Detector pixel pitch (mm) | Convert pixels → mm |
| `RTImageSID` / `RadiationMachineSAD` | Source-to-imager / source-to-axis distance (mm) | Magnification: a shift on the imager divided by SID/SAD gives the shift at isocenter |

The cell below lists your 14 files and shows the header of the first one.

In [ ]:
import pydicom

dcm_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.dcm")))
print(f"{len(dcm_files)} DICOM files:")
for f in dcm_files:
    print(" ", os.path.basename(f))

# Peek at the first file
d = pydicom.dcmread(dcm_files[0])
print("\n--- Header of the first image ---")
print(f"Modality              : {d.Modality}")
print(f"Manufacturer          : {d.Manufacturer}")
print(f"Radiation type        : {getattr(d, 'RTImageLabel', 'n/a')}")
print(f"Gantry angle [deg]    : {d.GantryAngle:.2f}")
print(f"Collimator angle [deg]: {d.BeamLimitingDeviceAngle:.2f}")
print(f"Couch angle [deg]     : {d.PatientSupportAngle:.2f}")
print(f"Pixel spacing [mm]    : {list(d.ImagePlanePixelSpacing)}")
print(f"SID / SAD [mm]        : {d.RTImageSID:.1f} / {d.RadiationMachineSAD}")
print(f"Magnification (SID/SAD): {d.RTImageSID/d.RadiationMachineSAD:.3f}")
print(f"Image shape (pixels)  : {d.pixel_array.shape}")
print(f"Pixel value range     : {d.pixel_array.min()} – {d.pixel_array.max()}")

### Inventory of all 14 fields

Let's cross-check against **Table 1** of the handout (the planned gantry / collimator / couch combinations). We'll pull the angles out of every DICOM and display them as a pandas table.

Note that 360.0 deg = 0.0 deg.

In [ ]:
import pandas as pd

def _clean(angle):
    """Round to 0.1 deg and wrap 360 -> 0 so the table matches Table 1."""
    a = round(angle, 1)
    return 0.0 if a == 360.0 else a

rows = []
for f in dcm_files:
    h = pydicom.dcmread(f, stop_before_pixels=True)
    rows.append({
        "file":       os.path.basename(f),
        "gantry":     _clean(h.GantryAngle),
        "collimator": _clean(h.BeamLimitingDeviceAngle),
        "couch":      _clean(h.PatientSupportAngle),
    })
pd.DataFrame(rows)

### Look at the raw image

A WL portal image is very simple: a **bright disk** (the 20 mm cone-collimated field) with a **small dark spot** in the middle (the tungsten BB absorbing the MV photons). Our whole job is to locate both centroids and measure the offset between them.

The cell below plots the first DICOM's pixel array. Try changing `IDX` to see different fields and notice that:

- The bright disk stays roughly centered on the EPID.
- The BB shifts by a fraction of a pixel as the gantry rotates — that tiny wobble is what we're after.

In [ ]:
import matplotlib.pyplot as plt

IDX = 0                      # change to 0..13 to explore

d   = pydicom.dcmread(dcm_files[IDX])
img = d.pixel_array

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))
ax1.imshow(img, cmap="gray")
ax1.set_title(f"Full EPID frame  ({img.shape[0]}×{img.shape[1]} px)")
ax1.set_xlabel("u (pixels)"); ax1.set_ylabel("v (pixels)")

cy, cx = img.shape[0] // 2, img.shape[1] // 2
half = 80
ax2.imshow(img[cy-half:cy+half, cx-half:cx+half], cmap="gray")
ax2.set_title("Zoom near center — bright field + dark BB")
ax2.set_xticks([]); ax2.set_yticks([])

fig.suptitle(f"Field #{IDX+1}: gantry={d.GantryAngle:.0f}°, "
             f"coll={d.BeamLimitingDeviceAngle:.0f}°, couch={d.PatientSupportAngle:.0f}°")
plt.show()

## 4. Run the Winston-Lutz analysis

Pylinac segments the radiation field and the BB on every image and solves for the 3D offset between the radiation isocenter and the BB using the Low *et al.* (1995) formalism.

### 🔧 TODO #1 — set the BB size

Our phantom uses a small tungsten sphere. Look at the lab handout / Winston-Lutz tool and set its diameter in millimeters.

In [ ]:
from pylinac import WinstonLutz

BB_SIZE_MM = ...      # TODO #1: BB diameter in mm

wl = WinstonLutz(DATA_DIR)
wl.analyze(bb_size_mm=BB_SIZE_MM)
print(f"Analyzed {len(wl.images)} images.")

## 5. Per-image 2D shifts (`Δu`, `Δv`, total)

`u` and `v` are the in-plane axes of the MV portal image (see **Figure 1** of the lab handout) — the horizontal and vertical directions on the EPID, expressed in mm at the isocenter plane. Pylinac doesn't use the `u`/`v` names; it just calls them `.x` and `.y` on the returned vector:

```python
img.cax2bb_vector.x   # horizontal shift on the imager  →  Δu
img.cax2bb_vector.y   # vertical   shift on the imager  →  Δv
img.cax2bb_distance   # sqrt(Δu² + Δv²), the total 2D shift
```

The sign convention is arbitrary (pylinac's `.y` is inverted compared to raw pixel coordinates), but the magnitude is what TG-142 cares about.

### 🔧 TODO #2 — complete the loop

Fill `du`, `dv`, and `total` using those attributes.

> **Hint:** `shift = img.cax2bb_vector` gives you `shift.x` and `shift.y`. For the total, use `img.cax2bb_distance`.

In [ ]:
print(f"{'#':>3} {'Gantry':>7} {'Coll':>6} {'Couch':>6} "
      f"{'du [mm]':>10} {'dv [mm]':>10} {'Total [mm]':>12}")
print("-" * 60)
for i, img in enumerate(wl.images, start=1):
    shift = img.cax2bb_vector       # 2D vector in the imager plane (mm at iso)

    du    = ...    # TODO #2: horizontal shift on the EPID
    dv    = ...     # TODO #2: vertical   shift on the EPID
    total = ...     # TODO #2: total 2D shift magnitude

    print(f"{i:>3} {img.gantry_angle:>7.1f} {img.collimator_angle:>6.1f} "
          f"{img.couch_angle:>6.1f} {du:>+10.4f} {dv:>+10.4f} {total:>12.4f}")

## 6. The 3D shift vector

A single image only sees the shift **projected onto the imager plane** — the component along the beam is invisible. Imaging the BB from many angles lets pylinac solve for the 3D offset between the radiation isocenter and the BB (`wl.bb_shift_vector`).

### 🔧 TODO #3 — compute the magnitude

Fill in the Euclidean length of the 3D vector.

> **Hint:** $|\vec{s}| = \sqrt{s_x^2 + s_y^2 + s_z^2}$

In [ ]:
s = wl.bb_shift_vector

import math
magnitude = ...     # TODO #3

print(f"X: {s.x:+.4f} mm")
print(f"Y: {s.y:+.4f} mm")
print(f"Z: {s.z:+.4f} mm")
print(f"|3D|: {magnitude:.4f} mm")
print()
print(wl.bb_shift_instructions())      # which way to push the couch to fix it

## 7. Pylinac plots 🎨

Pylinac ships with several built-in visualizations. Run the cells below.

### 7a. Single image — field CAX (green) vs. BB (red)

In [ ]:
IMG_INDEX = 0                      # try changing to 0..13
wl.images[IMG_INDEX].plot()

### 7b. All 14 fields at once

By default `plot_images()` only shows gantry-only fields (collimator = couch = 0). Since our plan mixes gantry, collimator, and couch rotations, we pass `axis=Axis.GBP_COMBO` to force *every* image into the grid.


In [ ]:
from pylinac.winston_lutz import Axis
wl.plot_images(axis=Axis.GBP_COMBO)

### 7c. Axis-grouped view (gantry / collimator / couch)

`plot_axis_images(axis=...)` shows only the images where that single axis was rotated (the others held at 0). Useful for seeing which axis contributes most to the isocenter wobble.

In [ ]:
wl.plot_axis_images(axis=Axis.GANTRY)
wl.plot_axis_images(axis=Axis.COLLIMATOR)
wl.plot_axis_images(axis=Axis.COUCH)

### 7d. 3D isocenter visualization

An interactive-style 3D plot of the BB, radiation isocenter sphere, and per-axis isocenters.

In [ ]:
wl.plot_location()

### 7e. Big summary figure

In [ ]:
wl.plot_summary()

### 7f. (Optional) Publish a full PDF report

In [ ]:
wl.publish_pdf("WL_report.pdf")
print("Saved WL_report.pdf")

## 8. Pylinac summary

In [ ]:
print(wl.results())